# 深度學習框架應用（大學部）｜主題 12：預訓練模型與遷移應用

這本筆記本配合投影片《主題 12：預訓練模型與遷移應用》。五題都直接用 Hugging Face Hub 上別人練好的模型：先看 tokenizer 怎麼拆字，再用 pipeline 一行做情感分析和零樣本分類、取句子向量算相似度、讓小型 GPT 接龍，最後比一比「凍結骨幹」和「全部微調」。**練習一～五，每題 20%，合計 100%**。

| 練習 | 題目 | 投影片（練習頁） | 觀念在哪一頁 |
|---|---|---|---|
| 一 | tokenizer 怎麼拆字：子詞、特殊符號、attention_mask | 第 120 頁 | 第 18～20、22、24 頁 |
| 二 | pipeline 做情感分析與零樣本分類 | 第 121 頁 | 第 52～54 頁 |
| 三 | 句子向量與餘弦相似度：平均池化 | 第 122 頁 | 第 55、56、60～62 頁 |
| 四 | GPT 接龍：溫度、greedy 與取樣 | 第 123 頁 | 第 41、44～47 頁 |
| 五 | 凍結骨幹 vs 全部微調（400 則影評） | 第 124 頁 | 第 65、66、73、75、76 頁 |

**怎麼用**

1. 用私人 Google 帳號登入（不使用學校帳號），按上方的「複製到雲端硬碟」（或「檔案 › 在雲端硬碟中儲存複本」）另存一份自己的複本，之後都改自己那份。**沒有另存就直接寫，分頁一關就全部不見。** 存好後先把檔名改成 `深度學習_W12_學號_姓名.ipynb`（左上角點檔名就能改）。
2. 先跑「環境準備」，再由上往下做；標「準備（直接執行）」的格子不用改，直接跑。每一題只有一格標 ✏️ 的程式格：把每個 `___` 換成答案再執行，提示寫在同一行後面的註解。還沒把 `___` 換掉就執行，會出現紅字錯誤（例如 TypeError、ValueError、KeyError、AttributeError），錯誤訊息裡箭頭 `---->` 指的那一行就是還沒填的空格，填好再跑一次就好。
3. 有 GPU 就用（「執行階段 › 變更執行階段類型」選 T4 GPU），沒有也沒關係：每一題用 CPU 都能在 10 分鐘內跑完（練習五最久，CPU 約 3～6 分鐘）。每一題第一次執行都會從 Hugging Face Hub 下載模型，請先確認網路；出現 `HF_TOKEN` 的提醒可以不理它。
4. 模型是別人練好的，**分數和生成的句子不一定和同學一模一樣**；「跑對了會看到」裡寫死的數字（token 數、形狀、參數量）才一定要一樣。
5. **每題做完就截一張整個螢幕的圖（截圖 ①～⑤）**，要看得到身份指紋：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間，以及這一題的輸出。**看不到身份指紋，那一題不計分。**

**繳交**：做完照最後一格「繳交前檢查」，把**分享連結 1 個、.ipynb 檔 1 個、截圖 5 張（①～⑤）**交到 Flipclass 作業區。

## 環境準備

五題都會用到下面這些套件、六個模型的名字（`CKPT`），以及數參數的小工具 `count()`。先跑這一格。

In [ ]:
# ===== 環境準備（直接執行）=====
%pip install -q datasets
import time, torch
import torch.nn.functional as F
import transformers
from transformers import (AutoTokenizer, AutoModel, pipeline,
                          AutoModelForSequenceClassification,
                          AutoModelForCausalLM, DataCollatorWithPadding)
from datasets import load_dataset
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
transformers.logging.set_verbosity_error()      # 只留錯誤訊息，畫面乾淨一點
dev = 'cuda' if torch.cuda.is_available() else 'cpu'   # 有 GPU 就用；沒有，CPU 也跑得完
print('PyTorch', torch.__version__, '｜transformers', transformers.__version__, '｜裝置', dev)

CKPT = {   # 這次用到的六個預訓練模型（都放在 Hugging Face Hub，第一次用到時才下載）
    'zh':   'google-bert/bert-base-chinese',                              # 中文 BERT：一個字一個 token
    'en':   'distilbert/distilbert-base-uncased',                         # 英文 DistilBERT：WordPiece 子詞
    'gpt':  'distilbert/distilgpt2',                                      # 小型 GPT-2：位元組 BPE、會接龍
    'sst2': 'distilbert/distilbert-base-uncased-finetuned-sst-2-english', # 已經微調好的情感分析模型
    'nli':  'typeform/distilbert-base-uncased-mnli',                      # 判斷兩句話關係（零樣本分類用）
    'sent': 'sentence-transformers/all-MiniLM-L6-v2',                     # 專門產生句子向量的小模型
}

def count(m, trainable=True):    # 數參數：預設只數「會被訓練」的（requires_grad=True）；trainable=False 就全部都數
    return sum(p.numel() for p in m.parameters() if p.requires_grad or not trainable)

## 練習一：tokenizer 怎麼拆字 (20%)

> 題目在投影片**第 120 頁**｜觀念在第 18～20、22、24 頁

**情境**：模型看不懂文字，只看得懂編號。tokenizer 負責把一句話切成一塊一塊的 token、補上特殊符號、查表換成編號，一批長短不一的句子還要補齊，再用 attention_mask 標出哪幾格是補的。這題用三個真的 tokenizer 切同樣的句子，比一比切法。

**這題在練**：用 `AutoTokenizer` 載入中文 BERT、英文 DistilBERT 和 GPT-2 的 tokenizer，看懂 input_ids、特殊符號、子詞的 `##`、補齊與 attention_mask

**步驟**

1. 先執行「練習一：準備」，它準備了四句長短不一的影評 `SENTS4`
2. 填好三個空格：[SEP] 的編號（看上一行印出的編號找）、「只切開、不轉編號」的方法、一批句子要不要補齊
3. 對一對：中文是不是一字一 token？英文 tokenizers 切成幾片？四句補齊後形狀多大？GPT-2 切中文為什麼特別多片？

**要繳交**

- 截圖 ①：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習一：準備（直接執行，不用改）=====
SENTS4 = ['這部電影的節奏很棒', '劇本太薄弱', '節奏明快看得很過癮', '演員演技在線但劇本太薄弱']   # 四句長短不一的影評
print('四句各幾個字：', [len(s) for s in SENTS4])

In [ ]:
# ✏️ 練習一：三種 tokenizer 怎麼拆字
zh = AutoTokenizer.from_pretrained(CKPT['zh'])
ids = zh('這部電影的節奏很棒')['input_ids']
print(len(ids), ids)
print(zh.convert_ids_to_tokens(ids))
print(zh.convert_ids_to_tokens([101, ___, 0]))  # ✏️ [SEP]
en = AutoTokenizer.from_pretrained(CKPT['en'])
print(en.___('tokenizers'), en.vocab_size)  # ✏️ 只切
b = zh(SENTS4, padding=___, return_tensors='pt')  # ✏️
print(tuple(b['input_ids'].shape))
print(b['attention_mask'])  # 1 真的字、0 補的
gt = AutoTokenizer.from_pretrained(CKPT['gpt'])
print(gt.tokenize('Hello world'))
print(gt.tokenize('這部電影'))

#### 📸 截圖 ①

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：全部輸出（token 數與編號、切開的 token、三個特殊符號、英文子詞與詞彙量、補齊後的形狀、4×14 的 attention_mask、GPT-2 的兩種切法）

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 中文 BERT 為什麼幾乎一個字一個 token？英文的 tokenizers 為什麼會切成 token、##izer、##s 三片？## 是什麼意思？**

（在這裡寫）

**2. 第二句「劇本太薄弱」的 attention_mask 為什麼是 7 個 1、7 個 0？GPT-2 為什麼把 4 個中文字切成 8 個 token？**

（在這裡寫）

## 練習二：pipeline 做情感分析與零樣本分類 (20%)

> 題目在投影片**第 121 頁**｜觀念在第 52～54 頁

**情境**：有時候根本不用自己訓練：Hub 上已經有人把 DistilBERT 微調成情感分析模型，pipeline 會把 tokenizer、模型、softmax、查標籤名字這幾步包在一起，一行就能用。零樣本分類更厲害：模型從來沒看過我們給的標籤，也能挑出最合適的那一個。

**這題在練**：用 `pipeline()` 做情感分析與零樣本分類，看懂輸出的 label、score，並知道 score 是怎麼來的

**步驟**

1. 先執行「練習二：準備」：四則英文影評 `REVIEWS`、一則新聞 `NEWS`、四個候選標籤 `LABELS`
2. 填好兩個空格：情感分析的任務名稱、零樣本分類要比較的候選標籤
3. 看每則影評被判成什麼、score 多少；再看新聞被分到哪一類，四個分數加起來是多少

> 第一次執行會下載兩個模型（各約 270 MB）。分數是別人模型算出來的，和同學差一點點是正常的。

**要繳交**

- 截圖 ②：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習二：準備（直接執行，不用改）=====
REVIEWS = ['An absolute delight from start to finish.',             # 四則英文影評
           'The plot is thin and the acting feels wooden.',
           'Not bad at all, I would happily watch it again.',
           'I wanted to like it, but it was a waste of two hours.']
NEWS = 'The new phone has a great camera, but the battery dies by noon.'   # 一則沒有標籤的新聞
LABELS = ['technology', 'sports', 'cooking', 'politics']                    # 候選標籤：模型從來沒有用它們訓練過
print(len(REVIEWS), '則影評｜', len(LABELS), '個候選標籤')

In [ ]:
# ✏️ 練習二：pipeline 做情感分析與零樣本分類
clf = pipeline('___',                     # ✏️ 任務名稱
               model=CKPT['sst2'])
print(clf.model.config.id2label)          # 編號 → 名字
for r, o in zip(REVIEWS, clf(REVIEWS)):
    print(f"{o['label']:8} {o['score']:.3f}  {r[:30]}")
zsc = pipeline(
    'zero-shot-classification', model=CKPT['nli'])
out = zsc(NEWS, candidate_labels=___)     # ✏️ 候選
for lab, s in zip(out['labels'], out['scores']):
    print(f'{lab:10} {s:.3f}')
print('加起來', round(sum(out['scores']), 3))

#### 📸 截圖 ②

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：id2label 那一行、四則影評的判斷與分數、新聞的四個標籤分數和「加起來」那一行

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 情感分析每一則的 score 為什麼一定在 0.5 以上？它是哪個數字經過什麼運算來的？**

（在這裡寫）

**2. 零樣本分類的模型從來沒有用 technology、sports 這些標籤訓練過，為什麼還能分類？四個分數為什麼加起來是 1？**

（在這裡寫）

## 練習三：句子向量與餘弦相似度 (20%)

> 題目在投影片**第 122 頁**｜觀念在第 55、56、60～62 頁

**情境**：骨幹模型（AutoModel）會替每個 token 輸出一個向量。把一句話裡「真的字」的向量平均起來，就得到一個代表整句話的句子向量；意思接近的句子，向量的方向也接近。語意搜尋、找重複的問題、零樣本分類，都是靠這一招。

**這題在練**：用 attention_mask 做平均池化（不讓補齊的格子混進來），把向量長度變成 1，再用矩陣乘法一次算出所有句子兩兩之間的餘弦相似度

**步驟**

1. 先執行「練習三：準備」：載入 all-MiniLM-L6-v2（只要骨幹）和五個英文句子 `SENTS`
2. 填好三個空格：除以每句真的字數、沿著哪一維把長度變成 1、矩陣要轉置
3. 看相似度表：對角線是多少？哪兩句最像？股市那一句和誰都不太像嗎？

> 平均時一定要乘上 attention_mask：補齊的格子也有向量，混進來會把句子向量拉偏。

**要繳交**

- 截圖 ③：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習三：準備（直接執行，不用改；第一次會下載約 90 MB）=====
stok = AutoTokenizer.from_pretrained(CKPT['sent'])
smodel = AutoModel.from_pretrained(CKPT['sent']).to(dev).eval()    # 只要骨幹（AutoModel），不要任務頭
print('句子向量模型的參數：', f'{count(smodel):,}')
SENTS = ['A man is playing a guitar.',                    # 0
         'Someone is strumming a guitar on stage.',       # 1
         'The chef is cooking pasta.',                    # 2
         'A woman is chopping vegetables in the kitchen.',# 3
         'The stock market fell sharply today.']          # 4

In [ ]:
# ✏️ 練習三：平均池化取句子向量，算相似度
enc = stok(SENTS, padding=True, return_tensors='pt')
enc = enc.to(dev)
with torch.no_grad():
    H = smodel(**enc).last_hidden_state  # (句, 字, 384)
m = enc['attention_mask'].unsqueeze(-1).float()
emb = (H * m).sum(1) / m.___(1)          # ✏️ 除以字數
emb = F.normalize(emb, dim=___)          # ✏️ 長度變 1
sim = emb @ emb.___                      # ✏️ 轉置
print(tuple(H.shape), '→', tuple(emb.shape))
print(sim.cpu().round(decimals=2))
sim.fill_diagonal_(-1)                   # 不跟自己比
i, j = divmod(sim.argmax().item(), len(SENTS))
print('最像的一對：', i, j)

#### 📸 截圖 ③

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：H 和句子向量的形狀、5×5 的相似度表、「最像的一對」那一行

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 平均池化為什麼要先乘上 attention_mask，再除以 m.sum(1)，而不是直接 H.mean(1)？**

（在這裡寫）

**2. 為什麼先把每個向量的長度變成 1，emb @ emb.T 算出來的就是餘弦相似度？對角線為什麼是 1？**

（在這裡寫）

## 練習四：GPT 接龍：溫度、greedy 與取樣 (20%)

> 題目在投影片**第 123 頁**｜觀念在第 41、44～47 頁

**情境**：GPT 這類解碼器模型一次只猜「下一個 token」：給它一段開頭，它替詞彙表裡 50,257 個 token 各打一個分數，選一個接上去，再把變長的句子丟回去猜下一個。怎麼「選」決定了句子的個性：每次都挑最高分（greedy）很穩但容易重複，照機率抽籤（取樣）比較有變化，溫度則調整抽籤有多大膽。

**這題在練**：用 softmax 親手算溫度對機率的影響，取出模型對「下一個 token」的前五名，再比較 greedy 和不同溫度的取樣

**步驟**

1. 先執行「練習四：準備」：載入 distilgpt2、把提示 `PROMPT` 換成編號，並準備好接龍用的 `gen()`
2. 填好三個空格：分數要除以什麼、取最後一格的 logits、greedy 要不要抽籤
3. 比一比：溫度變小時第一名的機率怎麼變？greedy 跑兩次一樣嗎？溫度 0.7 和 1.5 的句子有什麼不同？

> `gen()` 每次都先固定亂數種子，所以同一個設定重跑結果一樣；換個 seed 才會抽出不同的句子。

**要繳交**

- 截圖 ④：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習四：準備（直接執行，不用改；第一次會下載約 350 MB）=====
gtok = AutoTokenizer.from_pretrained(CKPT['gpt'])
gpt = AutoModelForCausalLM.from_pretrained(CKPT['gpt']).to(dev).eval()     # 會接龍的模型：骨幹＋語言模型頭
print('distilgpt2 的參數：', f'{count(gpt):,}')
PROMPT = 'Once upon a time, a little robot'
ids = gtok(PROMPT, return_tensors='pt').to(dev)
print('提示有', ids['input_ids'].shape[1], '個 token')

def gen(seed=0, **kw):
    """從 PROMPT 往後接 20 個 token，回傳新接上去的那一段文字。kw 交給 generate()（例如 do_sample、temperature）。"""
    torch.manual_seed(seed)                                                  # 固定亂數：同樣設定、同樣結果
    out = gpt.generate(**ids, max_new_tokens=20, min_new_tokens=20,
                       pad_token_id=gtok.eos_token_id, **kw)
    return gtok.decode(out[0, ids['input_ids'].shape[1]:])

In [ ]:
# ✏️ 練習四：溫度、greedy 與取樣
z = torch.tensor([2.0, 1.0, 0.1])       # 三個字的分數
for T in [0.5, 1.0, 2.0]:
    p = torch.softmax(z / ___, dim=-1)  # ✏️ 溫度
    print(f'T={T}', [round(v, 3) for v in p.tolist()])
with torch.no_grad():
    nxt = gpt(**ids).logits[0, ___]     # ✏️ 最後一格
top = torch.softmax(nxt, -1).topk(5)
print([gtok.decode(int(i)) for i in top.indices])
a, b = gen(do_sample=___), gen(do_sample=False)  # ✏️
print('greedy 兩次一樣？', a == b, '｜', a)
for T in [0.7, 1.5]:
    print(f'T={T}：', gen(do_sample=True, temperature=T))

#### 📸 截圖 ④

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：三個溫度的機率、下一個 token 的前五名、greedy 那一行、兩個溫度各自接出來的句子

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 分數 [2.0, 1.0, 0.1] 在溫度 0.5、1.0、2.0 時，第一名的機率各是多少？溫度變小、變大各會怎樣？**

（在這裡寫）

**2. greedy 為什麼跑兩次一模一樣？你的 T = 1.5 那一句有什麼問題？top-p（核取樣）又是什麼？**

（在這裡寫）

## 練習五：凍結骨幹 vs 全部微調 (20%)

> 題目在投影片**第 124 頁**｜觀念在第 65、66、73、75、76 頁

**情境**：同一個預訓練的 DistilBERT、同一份 400 則英文影評，比較兩種接法：①凍結骨幹，只訓練新加的分類頭；②整個模型一起微調，學習率要很小。兩種都只練 1 個 epoch，看可訓練參數差多少、分數差多少、時間差多少。

**這題在練**：用 `requires_grad` 凍結骨幹、用 `count()` 確認到底有幾個參數在學，並體會微調為什麼要用很小的學習率

**步驟**

1. 先執行「練習五：準備」：下載 rotten_tomatoes 影評、抽出訓練 400 則與驗證 200 則、用 `map(batched=True)` 切好字，並準備 `fit()` 與 `evaluate()`
2. 填好四個空格：全部微調的學習率是幾 × 1e-5（微調用 2e-5～5e-5；資料少、只練一輪，填 5）、幾類、骨幹叫什麼、凍結要設成什麼
3. 比一比兩行：可訓練參數各多少、占全部幾 %；驗證的 accuracy、macro-F1；各花了幾秒

> 練習五最花時間：用 GPU 大約 1 分鐘內；只用 CPU 約 3～6 分鐘，請耐心等它跑完。

**要繳交**

- 截圖 ⑤：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習五：準備（直接執行，不用改；第一次會下載資料與約 270 MB 的模型）=====
ds = load_dataset('cornell-movie-review-data/rotten_tomatoes')        # 英文短影評：label 0 負面、1 正面
EN = CKPT['en']                                                       # 英文 DistilBERT
tok = AutoTokenizer.from_pretrained(EN)
def encode(batch):                                                    # 一次一批：截到 64 個 token，先不補齊
    return tok(batch['text'], truncation=True, max_length=64)
train_ds = ds['train'].shuffle(seed=0).select(range(400)).map(encode, batched=True, remove_columns=['text'])   # 小資料：訓練 400 則
val_ds = ds['validation'].shuffle(seed=0).select(range(200)).map(encode, batched=True, remove_columns=['text'])  # 驗證 200 則
collate = DataCollatorWithPadding(tok)                                # 組批次時才補齊，只補到這一批最長
Cls = AutoModelForSequenceClassification                              # 名字太長，取個小名：骨幹＋分類頭
g = torch.Generator().manual_seed(0)                                  # 固定洗牌順序
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate, generator=g)
val_dl = DataLoader(val_ds, batch_size=64, collate_fn=collate)
print('訓練', len(train_ds), '則、驗證', len(val_ds), '則｜一個 epoch', len(train_dl), '步')

def evaluate(model):
    """在驗證集上算 accuracy 和 macro-F1。"""
    model.eval(); ys, ps = [], []
    with torch.no_grad():
        for batch in val_dl:
            batch = {k: v.to(dev) for k, v in batch.items()}
            ps += model(**batch).logits.argmax(-1).tolist()
            ys += batch['labels'].tolist()
    acc = sum(p == y for p, y in zip(ps, ys)) / len(ys)
    return acc, f1_score(ys, ps, average='macro')

def fit(model, lr, epochs=1):
    """訓練 epochs 輪（只更新 requires_grad=True 的參數），回傳 (val accuracy, val macro-F1, 訓練秒數)。"""
    model.to(dev).train()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    t0 = time.time()
    for _ in range(epochs):
        for batch in train_dl:
            batch = {k: v.to(dev) for k, v in batch.items()}   # 有 labels，模型自己算 loss
            loss = model(**batch).loss
            opt.zero_grad(); loss.backward(); opt.step()
    sec = time.time() - t0
    return (*evaluate(model), sec)

In [ ]:
# ✏️ 練習五：凍結骨幹 vs 全部微調
for mode in ['凍結', '全部']:
    lr = 1e-3 if mode == '凍結' else ___ * 1e-5  # ✏️
    torch.manual_seed(0)  # 新頭的起點固定
    model = Cls.from_pretrained(EN, num_labels=___)  # ✏️
    if mode == '凍結':
        for p in model.___.parameters():         # ✏️ 骨幹
            p.requires_grad = ___                # ✏️
    n, N = count(model), count(model, False)
    acc, f1, sec = fit(model, lr)
    print(f'{mode}：可訓練 {n:,}（{n / N:.2%}）'
          f'  acc {acc:.3f}  F1 {f1:.3f}  {sec:.0f} 秒')

#### 📸 截圖 ⑤

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：準備格印出的筆數與步數，以及「凍結」「全部」兩行結果

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 「凍結」那一行的可訓練參數 592,130 是怎麼來的？為什麼不是 768 × 2 + 2 = 1,538？**

（在這裡寫）

**2. 兩種接法的分數和時間差多少？為什麼全部微調的學習率要用 5e-5，凍結只訓練頭卻可以用 1e-3？**

（在這裡寫）

## AI 使用紀錄

有用 AI 就寫下來，沒用就寫「沒有使用」。

| 用了哪個 AI | 問了什麼（大概就好） | 它哪裡幫到你 | 它哪裡說錯、你怎麼發現的 |
|---|---|---|---|
| （例：ChatGPT） | （例：attention_mask 是做什麼的） |  |  |

## 繳交前檢查

1. **重跑一次**：「執行階段 › 重新啟動工作階段並執行所有儲存格」，每一格都有輸出、沒有紅字。
2. **檔名**：`深度學習_W12_學號_姓名.ipynb`（左上角點檔名就能改）。
3. **截圖**：5 張（①～⑤），每張都是整個螢幕，看得到身份指紋（左上的檔名、右上的 Google 頭貼、右下工作列的日期時間）和那一題的輸出。
4. **開放分享**：右上角「分享」→「一般存取權」改成「知道連結的任何人」→ 角色選「檢視者」→「複製連結」。
5. **自己檢查**：開一個無痕視窗貼上連結，不用登入就打得開才算成功。
6. **下載**：「檔案 › 下載 › 下載 .ipynb」。
7. **交到 Flipclass**：分享連結 1 個、.ipynb 檔 1 個、截圖 5 張（①～⑤），交到 Flipclass 作業區，期限照 Flipclass 規定。